# DSPy — Otimização com `COPRO`

Neste notebook será demonstrado o uso do otimizador `COPRO` do DSPy em um problema de **classificação binária de textos**.

Será utilizada a base **Natural Language Processing with Disaster Tweets**, disponibilizada no Kaggle. O objetivo é classificar cada tweet em uma das seguintes categorias:

* `0`: o tweet **não descreve um desastre real**;
* `1`: o tweet **descreve um desastre real**.

O experimento será dividido em duas etapas:

1. avaliar um classificador DSPy utilizando a instrução original definida na `Signature`;
2. utilizar o `COPRO` para procurar automaticamente instruções alternativas e selecionar aquela que obtiver melhor desempenho no conjunto utilizado durante a otimização.

Diferentemente de otimizadores baseados em demonstrações few-shot, como `LabeledFewShot`, `BootstrapFewShot` ou `KNNFewShot`, o `COPRO` atua principalmente sobre a **instrução textual do programa**.

O otimizador gera diferentes versões da instrução, avalia cada uma delas utilizando uma métrica e utiliza os resultados para produzir novas candidatas.

Ao final, a melhor versão encontrada é utilizada como programa otimizado.

A avaliação final continuará utilizando:

* Accuracy;
* Precision;
* Recall;
* F1-score.

A métrica principal para comparar o programa original e o programa otimizado será o **F1-score**.

No DSPy, COPRO significa Coordinate Ascent Prompt Optimization — algo como Otimização de Prompts por Ascensão de Coordenadas.

In [1]:
import os
from dotenv import load_dotenv  # Carrega variáveis de ambiente do arquivo .env
import dspy  # Framework para otimização de prompts com Language Models
import pandas as pd

from typing import Literal
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

from tqdm.auto import tqdm

/home/leonardo/Documentos/github/dspy_studies/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()  # Lê variáveis do arquivo .env

True

## Setup - Configuração do Modelo

Primeiro, carregamos as variáveis de ambiente (como API key) do arquivo `.env` na raiz do projeto. Isso evita hardcoding de credenciais no código.

In [3]:
lm = dspy.LM(
    "openai/gpt-5-mini",  # Modelo OpenAI GPT-5 Mini (modelo compacto e rápido)
    api_key=os.getenv("OPENAI_API_KEY"),  # API key carregada da variável de ambiente
)

# Modelo utilizado pelo COPRO para gerar novas instruções candidatas
prompt_lm = dspy.LM(
    "openai/gpt-4.1-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
)

# Configura o modelo padrão para todas as operações DSPy
dspy.configure(lm=lm)

## 3. Leitura da base de dados

Será utilizada a base da competição **Natural Language Processing with Disaster Tweets**, do Kaggle.

Referência:

https://www.kaggle.com/competitions/nlp-getting-started

Para este experimento são relevantes principalmente duas colunas:

| Coluna   | Descrição                   |
| -------- | --------------------------- |
| `text`   | Texto do tweet              |
| `target` | Classe correta (`0` ou `1`) |

O problema consiste, portanto, em aprender a relação:

`text → target`


In [4]:
df = pd.read_csv('disaster_tweets.csv')
df = df.head(200)
df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


## Análise da distribuição das classes

Antes da divisão dos dados, é importante verificar quantos exemplos existem de cada classe.

Além da quantidade absoluta, será analisada a proporção entre tweets classificados como `0` e `1`.

Essa análise é importante porque o **F1-score** considera conjuntamente precisão e recall e é especialmente útil quando existe algum grau de desbalanceamento entre as classes.


In [5]:
df["target"].value_counts()

target
0    103
1     97
Name: count, dtype: int64

In [6]:
df["target"].value_counts(normalize=True)

target
0    0.515
1    0.485
Name: proportion, dtype: float64

## Separação entre treino e teste

A base será dividida em:

* **80% para treinamento/otimização**;
* **20% para teste**.

O parâmetro `stratify=df["target"]` é utilizado para manter aproximadamente a mesma proporção entre as classes `0` e `1` nos dois conjuntos.

O conjunto de **treino** será utilizado pelo `COPRO` durante a otimização das instruções.

Para cada instrução candidata, o programa será executado sobre os exemplos desse conjunto e seu desempenho será calculado utilizando a métrica fornecida ao otimizador.

O conjunto de **teste**, por outro lado, será mantido separado e utilizado exclusivamente para comparar o programa original com o programa otimizado.

Dessa forma, os exemplos utilizados para selecionar a melhor instrução não são utilizados na avaliação final.

In [7]:
df_train, df_test = train_test_split(
    df[["text", "target"]],
    test_size=0.20,
    random_state=42,
    stratify=df["target"],
)

print(f"Treino: {len(df_train)} exemplos")
print(f"Teste:  {len(df_test)} exemplos")

Treino: 160 exemplos
Teste:  40 exemplos


## Conversão para `dspy.Example`

O DSPy representa exemplos de treino e teste por meio da classe `dspy.Example`.

Neste problema, cada exemplo possui dois campos:

* `text`: entrada fornecida ao modelo;
* `target`: resposta esperada.

A chamada:

`with_inputs("text")`

informa explicitamente ao DSPy que `text` deve ser utilizado como entrada do programa.

Consequentemente, `target` passa a ser tratado como o **label**, ou seja, a resposta esperada para aquele exemplo.

Conceitualmente, cada registro passa a ter a seguinte estrutura:

`entrada: text → saída esperada: target`

In [8]:
def dataframe_para_dspy(dataframe):
    exemplos = []

    for _, row in dataframe.iterrows():
        exemplo = dspy.Example(
            text=row["text"],
            target=int(row["target"]),
        ).with_inputs("text")

        exemplos.append(exemplo)

    return exemplos

In [9]:
trainset = dataframe_para_dspy(df_train)
testset = dataframe_para_dspy(df_test)

print(f"Trainset DSPy: {len(trainset)}")
print(f"Testset DSPy:  {len(testset)}")

Trainset DSPy: 160
Testset DSPy:  40


In [10]:
trainset[0]

Example({'text': 'Accident center lane blocked in #SantaClara on US-101 NB before Great America Pkwy #BayArea #Traffic http://t.co/pmlOhZuRWR', 'target': 1}) (input_keys={'text'})

## Definição da tarefa com uma `Signature`

No DSPy, uma `Signature` descreve declarativamente a tarefa que será executada pelo modelo.

A `ClassificarTweet` possui:

* um `InputField` chamado `text`, contendo o tweet;
* um `OutputField` chamado `target`, contendo a classificação.

O tipo:

`Literal[0, 1]`

restringe a resposta esperada às duas classes válidas do problema.

Dessa forma, a Signature define claramente o contrato:

`texto do tweet → 0 ou 1`


In [11]:
class ClassificarTweet(dspy.Signature):
    """
    Determine se o tweet descreve um desastre real.

    Retorne:
    - 1 se o tweet estiver relacionado a um desastre real.
    - 0 caso contrário.
    """

    text: str = dspy.InputField(
        desc="Texto do tweet que deve ser classificado."
    )

    target: Literal[0, 1] = dspy.OutputField(
        desc="1 para desastre real e 0 para não desastre."
    )

In [12]:
classificador_base = dspy.Predict(ClassificarTweet)

In [13]:
def avaliar_classificador(programa, dataset, descricao="Avaliando"):
    """
    Executa um programa DSPy sobre um dataset e calcula
    métricas globais de classificação.
    """

    y_true = []
    y_pred = []

    for exemplo in tqdm(dataset, desc=descricao):

        # Executa o programa utilizando somente os campos
        # marcados como entrada pelo with_inputs(...)
        predicao = programa(**exemplo.inputs())

        # Label verdadeiro
        y_true.append(int(exemplo.target))

        # Label previsto pelo DSPy
        y_pred.append(int(predicao.target))

    # Calcula as métricas sobre todo o conjunto
    resultado = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "y_true": y_true,
        "y_pred": y_pred,
    }

    return resultado

In [14]:
resultado_base = avaliar_classificador(
    classificador_base,
    testset,
    descricao="Baseline zero-shot",
)

Baseline zero-shot: 100%|█| 40/40 [00:0


## Avaliação do baseline

O classificador inicial será executado sobre todos os exemplos do conjunto de teste.

Para cada exemplo:

1. o campo `text` é enviado ao programa;
2. o programa produz uma previsão para `target`;
3. a previsão é comparada com o `target` verdadeiro.

Ao final são calculadas as métricas globais de classificação.

Esse resultado será considerado o desempenho **antes da otimização**.


In [15]:
print(f"F1 baseline: {resultado_base['f1']:.4f}")

F1 baseline: 0.9474


In [16]:
print(f"Accuracy:  {resultado_base['accuracy']:.4f}")
print(f"Precision: {resultado_base['precision']:.4f}")
print(f"Recall:    {resultado_base['recall']:.4f}")
print(f"F1:        {resultado_base['f1']:.4f}")

Accuracy:  0.9500
Precision: 0.9474
Recall:    0.9474
F1:        0.9474


## Métrica utilizada durante a otimização

O `COPRO` precisa de uma métrica para comparar as diferentes instruções candidatas.

Essa métrica é chamada individualmente para cada exemplo do conjunto utilizado durante a compilação e deve retornar um valor numérico que indique a qualidade da previsão.

Neste problema de classificação será utilizada uma métrica de **acerto exato**:

```python
def metrica_copro(example, prediction, trace=None):
    return int(example.target) == int(prediction.target)

In [17]:
def metrica_copro(example, prediction, trace=None):
    """
    Métrica utilizada internamente pelo COPRO.

    Retorna 1 quando a classificação está correta
    e 0 quando está incorreta.
    """
    return int(example.target) == int(prediction.target)

## Otimização com `COPRO`

O `COPRO` (**Coordinate Ascent Prompt Optimization**) é um otimizador do DSPy voltado para a otimização das **instruções utilizadas pelos módulos do programa**.

Diferentemente do `KNNFewShot`, o `COPRO` não procura exemplos semelhantes e não adiciona demonstrações dinamicamente.

Em vez disso, ele gera diferentes versões da instrução definida na `Signature`, executa o programa com essas instruções sobre o conjunto de treinamento e compara seus resultados utilizando uma métrica.

O funcionamento pode ser representado conceitualmente como:

```text
instrução original
       ↓
geração de várias instruções candidatas
       ↓
avaliação de cada candidata no trainset
       ↓
pontuação fornecida pela métrica
       ↓
seleção das melhores candidatas
       ↓
nova rodada de propostas
       ↓
melhor instrução encontrada

In [18]:
BREADTH = 3
DEPTH = 2

optimizer = dspy.COPRO(
    prompt_model=prompt_lm,       # Modelo responsável por gerar novas instruções
    metric=metrica_copro,         # Métrica usada para avaliar cada candidata
    breadth=BREADTH,              # Número de candidatas exploradas por rodada
    depth=DEPTH,                  # Número de rodadas de otimização
    init_temperature=1.4,         # Diversidade na geração das propostas
    track_stats=True,             # Armazena estatísticas da otimização
)

## Compilação do programa

A compilação do `COPRO` recebe:

* o programa que será otimizado (`student`);
* o conjunto utilizado para avaliar as instruções candidatas (`trainset`).

```python
classificador_otimizado = optimizer.compile(
    student=classificador_base,
    trainset=trainset,
)

In [19]:
classificador_otimizado = optimizer.compile(
    student=classificador_base,
    trainset=trainset,
    eval_kwargs={
        "num_threads": 4,
        "display_progress": True,
        "display_table": False,
    },
)

2026/09/07 20:02:17 INFO dspy.teleprompt.copro_optimizer: Iteration Depth: 1/2.






[2026-09-07T20:02:17.629753]

System message:

Your input fields are:
1. `basic_instruction` (str): The initial instructions before optimization
Your output fields are:
1. `proposed_instruction` (str): The improved instructions for the language model
2. `proposed_prefix_for_output_field` (str): The string at the end of the prompt, which will help the model start solving the task
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## basic_instruction ## ]]
{basic_instruction}

[[ ## proposed_instruction ## ]]
{proposed_instruction}

[[ ## proposed_prefix_for_output_field ## ]]
{proposed_prefix_for_output_field}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are an instruction optimizer for large language models. I will give you a ``signature`` of fields (inputs and outputs) in English. Your task is to propose an instruction that will lead a good language model to perform the task well. Don't be 

2026/09/07 20:02:17 INFO dspy.teleprompt.copro_optimizer: At Depth 1/2, Evaluating Prompt Candidate #1/3 for Predictor 1 of 1.


Average Metric: 149.00 / 160 (93.1%): 1

2026/09/07 20:02:18 INFO dspy.evaluate.evaluate: Average Metric: 149 / 160 (93.1%)
2026/09/07 20:02:18 INFO dspy.teleprompt.copro_optimizer: At Depth 1/2, Evaluating Prompt Candidate #2/3 for Predictor 1 of 1.


...






[2026-09-07T20:02:19.165323]

System message:

Your input fields are:
1. `attempted_instructions` (str):
Your output fields are:
1. `proposed_instruction` (str): The improved instructions for the language model
2. `proposed_prefix_for_output_field` (str): The string at the end of the prompt, which will help the model start solving the task
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## attempted_instructions ## ]]
{attempted_instructions}

[[ ## proposed_instruction ## ]]
{proposed_instruction}

[[ ## proposed_prefix_for_output_field ## ]]
{proposed_prefix_for_output_field}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are an instruction optimizer for large language models. I will give some task instructions I've tried, along with their corresponding validation scores. The instructions are arranged in increasing order based on their scores, where higher scores indicate better quality.


2026/09/07 20:02:20 INFO dspy.evaluate.evaluate: Average Metric: 145 / 160 (90.6%)
2026/09/07 20:02:20 INFO dspy.teleprompt.copro_optimizer: At Depth 2/2, Evaluating Prompt Candidate #3/3 for Predictor 1 of 1.







[2026-09-07T20:02:19.165323]

System message:

Your input fields are:
1. `attempted_instructions` (str):
Your output fields are:
1. `proposed_instruction` (str): The improved instructions for the language model
2. `proposed_prefix_for_output_field` (str): The string at the end of the prompt, which will help the model start solving the task
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## attempted_instructions ## ]]
{attempted_instructions}

[[ ## proposed_instruction ## ]]
{proposed_instruction}

[[ ## proposed_prefix_for_output_field ## ]]
{proposed_prefix_for_output_field}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are an instruction optimizer for large language models. I will give some task instructions I've tried, along with their corresponding validation scores. The instructions are arranged in increasing order based on their scores, where higher scores indicate better quality.


2026/09/07 20:02:20 INFO dspy.evaluate.evaluate: Average Metric: 141 / 160 (88.1%)







[2026-09-07T20:02:19.165323]

System message:

Your input fields are:
1. `attempted_instructions` (str):
Your output fields are:
1. `proposed_instruction` (str): The improved instructions for the language model
2. `proposed_prefix_for_output_field` (str): The string at the end of the prompt, which will help the model start solving the task
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## attempted_instructions ## ]]
{attempted_instructions}

[[ ## proposed_instruction ## ]]
{proposed_instruction}

[[ ## proposed_prefix_for_output_field ## ]]
{proposed_prefix_for_output_field}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are an instruction optimizer for large language models. I will give some task instructions I've tried, along with their corresponding validation scores. The instructions are arranged in increasing order based on their scores, where higher scores indicate better quality.


## Inspeção do resultado da otimização

Ao contrário do `KNNFewShot`, não existem vizinhos ou demonstrações recuperadas dinamicamente para inspecionar.

No `COPRO`, é mais interessante observar:

1. a instrução original;
2. a instrução escolhida pelo programa otimizado;
3. as instruções candidatas avaliadas;
4. as respectivas pontuações.

Isso permite visualizar diretamente o espaço de prompts explorado pelo otimizador.

In [20]:
print("=== INSTRUÇÃO ORIGINAL ===")
print(classificador_base.signature.instructions)

print("\n=== INSTRUÇÃO OTIMIZADA ===")
print(classificador_otimizado.signature.instructions)

=== INSTRUÇÃO ORIGINAL ===
Determine se o tweet descreve um desastre real.

Retorne:
- 1 se o tweet estiver relacionado a um desastre real.
- 0 caso contrário.

=== INSTRUÇÃO OTIMIZADA ===
Analise o conteúdo do tweet fornecido para determinar se ele descreve um desastre real. Seu objetivo é classificar o tweet da seguinte maneira:
- Retorne 1, se o tweet estiver relacionado a um desastre real que ocorreu.
- Retorne 0, caso contrário, inclusive se for um desastre fictício, informação não relacionada, comentário, ou notícia falsa.

Seja claro e conciso na sua decisão, baseando-se somente no conteúdo apresentado.


In [21]:
print(
    "Número de avaliações de candidatos:",
    classificador_otimizado.total_calls,
)

Número de avaliações de candidatos: 6


In [22]:
for i, candidato in enumerate(
    classificador_otimizado.candidate_programs[:5],
    start=1,
):
    print(f"--- Candidato {i} ---")
    print(f"Score: {candidato['score']:.4f}")
    print(f"Instrução: {candidato['instruction']}")
    print(f"Prefixo: {candidato['prefix']}")
    print()

--- Candidato 1 ---
Score: 93.1200
Instrução: Analise o conteúdo do tweet fornecido para determinar se ele descreve um desastre real. Seu objetivo é classificar o tweet da seguinte maneira:
- Retorne 1, se o tweet estiver relacionado a um desastre real que ocorreu.
- Retorne 0, caso contrário, inclusive se for um desastre fictício, informação não relacionada, comentário, ou notícia falsa.

Seja claro e conciso na sua decisão, baseando-se somente no conteúdo apresentado.
Prefixo: Classificação (1 para desastre real, 0 caso contrário):

--- Candidato 2 ---
Score: 92.5000
Instrução: Determine se o tweet descreve um desastre real.

Retorne:
- 1 se o tweet estiver relacionado a um desastre real.
- 0 caso contrário.
Prefixo: Target:

--- Candidato 3 ---
Score: 91.8800
Instrução: Você receberá um tweet que pode conter informações sobre um desastre. Sua tarefa é avaliar cuidadosamente o conteúdo do tweet e determinar se ele descreve um desastre que realmente ocorreu. Retorne "1" se o tweet se 

In [23]:
resultado_otimizado = avaliar_classificador(
    classificador_otimizado,
    testset,
    descricao="COPRO",
)

COPRO: 100%|█| 40/40 [00:00<00:00, 83.2


## Avaliação após a aplicação do `COPRO`

O programa otimizado pelo `COPRO` será avaliado utilizando **exatamente o mesmo conjunto de teste utilizado pelo baseline**.

Isso permite comparar:

* o classificador utilizando a instrução original;
* o classificador utilizando a instrução selecionada pelo `COPRO`.

O conjunto de teste não foi utilizado durante a busca pelas instruções candidatas.

Para cada exemplo do `testset`, o programa otimizado executará a classificação utilizando a nova instrução selecionada durante a compilação.

Ao final serão calculados novamente:

* Accuracy;
* Precision;
* Recall;
* F1-score.

Embora o `COPRO` utilize internamente uma métrica de acerto por exemplo para selecionar as instruções, o **F1-score global** continua sendo utilizado como métrica principal para comparar o programa original com o programa otimizado.

In [24]:
comparacao = pd.DataFrame(
    {
        "Modelo": [
            "Baseline (instrução original)",
            "COPRO",
        ],
        "Accuracy": [
            resultado_base["accuracy"],
            resultado_otimizado["accuracy"],
        ],
        "Precision": [
            resultado_base["precision"],
            resultado_otimizado["precision"],
        ],
        "Recall": [
            resultado_base["recall"],
            resultado_otimizado["recall"],
        ],
        "F1": [
            resultado_base["f1"],
            resultado_otimizado["f1"],
        ],
    }
)

comparacao

,Modelo,Accuracy,Precision,Recall,F1
0,Baseline (instrução original),0.95,0.947368,0.947368,0.947368
1,COPRO,0.95,1.000000,0.894737,0.944444


In [25]:
print("BASELINE")
print(
    classification_report(
        resultado_base["y_true"],
        resultado_base["y_pred"],
        digits=4,
    )
)

BASELINE
              precision    recall  f1-score   support

           0     0.9524    0.9524    0.9524        21
           1     0.9474    0.9474    0.9474        19

    accuracy                         0.9500        40
   macro avg     0.9499    0.9499    0.9499        40
weighted avg     0.9500    0.9500    0.9500        40



In [26]:
print(
    f"COPRO (breadth={BREADTH}, depth={DEPTH})"
)

print(
    classification_report(
        resultado_otimizado["y_true"],
        resultado_otimizado["y_pred"],
        digits=4,
    )
)

COPRO (breadth=3, depth=2)
              precision    recall  f1-score   support

           0     0.9130    1.0000    0.9545        21
           1     1.0000    0.8947    0.9444        19

    accuracy                         0.9500        40
   macro avg     0.9565    0.9474    0.9495        40
weighted avg     0.9543    0.9500    0.9497        40



## Salvando o programa otimizado pelo `COPRO`

O `COPRO` otimiza principalmente a instrução e outras informações associadas à `Signature` do programa.

Não existe, neste caso, um mecanismo dinâmico de recuperação de exemplos que precise ser serializado junto com a arquitetura.

Por isso, neste experimento será utilizado o **State-only Saving** do DSPy.

```python
classificador_otimizado.save(
    "COPRO.json",
    save_program=False,
)

In [27]:
classificador_otimizado.save(
    "COPRO.json",
    save_program=False,
)

In [28]:
# Recria a mesma arquitetura do programa
classificador_carregado = dspy.Predict(ClassificarTweet)

# Carrega o estado otimizado pelo COPRO
classificador_carregado.load(
    "COPRO.json"
)
classificador_carregado

Predict(StringSignature(text -> target
    instructions='Analise o conteúdo do tweet fornecido para determinar se ele descreve um desastre real. Seu objetivo é classificar o tweet da seguinte maneira:\n- Retorne 1, se o tweet estiver relacionado a um desastre real que ocorreu.\n- Retorne 0, caso contrário, inclusive se for um desastre fictício, informação não relacionada, comentário, ou notícia falsa.\n\nSeja claro e conciso na sua decisão, baseando-se somente no conteúdo apresentado.'
    text = Field(annotation=str required=True json_schema_extra={'desc': 'Texto do tweet que deve ser classificado.', '__dspy_field_type': 'input', 'prefix': 'Text:'})
    target = Field(annotation=Literal[0, 1] required=True json_schema_extra={'desc': '1 para desastre real e 0 para não desastre.', '__dspy_field_type': 'output', 'prefix': 'Classificação (1 para desastre real, 0 caso contrário):'})
))

In [29]:
tweet = "My phone battery died right before the meeting, what a disaster!"

predicao = classificador_carregado(
    text=tweet
)

print(predicao)

Prediction(
    target=0
)


In [30]:
# Mostra última chamada ao modelo (n=1 significa 1 última chamada)
dspy.inspect_history(n=1)





[2026-09-07T20:02:21.546662]

System message:

Your input fields are:
1. `text` (str): Texto do tweet que deve ser classificado.
Your output fields are:
1. `target` (Literal[0, 1]): 1 para desastre real e 0 para não desastre.
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## text ## ]]
{text}

Outputs will be a JSON object with the following fields.

{
  "target": "{target}        # note: the value you produce must exactly match (no extra characters) one of: 0; 1"
}
In adhering to this structure, your objective is: 
        Analise o conteúdo do tweet fornecido para determinar se ele descreve um desastre real. Seu objetivo é classificar o tweet da seguinte maneira:
        - Retorne 1, se o tweet estiver relacionado a um desastre real que ocorreu.
        - Retorne 0, caso contrário, inclusive se for um desastre fictício, informação não relacionada, comentário, ou notícia falsa.
       